# Lazy Browse — DestinE Climate DT Portfolio

This notebook provides an **instant xarray view** of DestinE Climate DT Generation 2 data.
Variables and coordinates appear immediately, actual data is fetched from the data bridge **only when you access values** (e.g. plotting, `.values`, `.compute()`).

**Prerequisites:** run `01_key_destine_once.ipynb` once to authenticate.

In [ ]:
import logging, warnings
import earthkit.data

# Cache & logging configuration
earthkit.data.config.set("cache-policy", "temporary")
earthkit.data.config.set("maximum-cache-size", "2G")
for _ln in ("earthkit.data", "polytope", "polytope.api", "polytope_client", "urllib3"):
    logging.getLogger(_ln).setLevel(logging.WARNING)
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [ ]:
import numpy as np
import pandas as pd
from polytope_zarr import PolytopeZarrStore
from destine_portfolio import PORTFOLIO

## 1. Define the dataset

We declare which coordinates, variables, and Polytope request fields to use.
Nothing is downloaded here — the store just builds metadata.

In [ ]:
# ── Configuration ──────────────────────────────────────────────────
MODELS       = ["ICON", "IFS-FESOM", "IFS-NEMO"]
EXPERIMENT   = "hist"              # 'hist', 'SSP3-7.0'
RESOLUTION   = "standard"          # nside=128 → 196 608 cells
N_CELLS      = 12 * 128**2         # HEALPix grid size
YEARS        = range(1990, 2015)   # 30-year historical period

# ── Choose a levtype (uncomment one) ──────────────────────────────
LEVTYPE = "sfc"                    # 34 vars — surface atmosphere
# LEVTYPE = "pl"                   #  9 vars — pressure levels (19 levels, 1000–1 hPa)
# LEVTYPE = "hl"                   #  2 vars — height levels (100 m, IFS-only)
# LEVTYPE = "sol"                  #  2 vars — soil / snow (5 levels)
# LEVTYPE = "o2d"                  # 13 vars — 2-D ocean & sea ice
# LEVTYPE = "o3d"                  #  5 vars — 3-D ocean (up to 75 levels)

# Polytope address per model (IFS-NEMO uses MN5, others use LUMI)
ADDRESS = {
    "ICON":       "polytope.lumi.apps.dte.destination-earth.eu",
    "IFS-FESOM":  "polytope.lumi.apps.dte.destination-earth.eu",
    "IFS-NEMO":   "polytope.mn5.apps.dte.destination-earth.eu",
}

# Infer activity from experiment
ACTIVITY = "baseline" if EXPERIMENT in ("hist", "cont") else "projections"

# ── Build coords from portfolio ────────────────────────────────────
lt = PORTFOLIO[LEVTYPE]

# Monthly time axis: one timestamp per month for each year
time_axis = pd.date_range(
    f"{min(YEARS)}-01", f"{max(YEARS)}-12", freq="MS"
)

coords = {
    "time":  time_axis,
    "cell":  range(N_CELLS),
    "model": MODELS,
}
# Add level dimension when the levtype requires it
if lt["levels"] is not None:
    coords["level"] = lt["levels"]

store = PolytopeZarrStore(
    address=ADDRESS,
    collection="destination-earth",
    base_request={
        "class": "d1",
        "dataset": "climate-dt",
        "type": "fc",
        "expver": "0001",
        "generation": "2",
        "realization": "1",
        "activity": ACTIVITY,
        "experiment": EXPERIMENT,
        "levtype": lt["levtype"],
        "resolution": RESOLUTION,
        "stream": "clmn",
    },
    coords=coords,
    variables=lt["variables"],
    internal_dims=["cell"],
    time_fields=["year", "month"],
)
print(store)

## 2. Open as xarray Dataset

This is **instant** — no data downloaded yet. You see all variables,
dimensions, and coordinate values.

In [ ]:
ds = store.open()
ds

## 3. Plot a single field (triggers lazy fetch)

Only now does the store actually call Polytope — fetching data for the
selected model, time, and variable.

In [ ]:
import healpy as hp
import matplotlib.pyplot as plt

field = ds["avg_2t"].sel(model="ICON", time="2014-06-01")
print(f"Fetching {field.sizes}...")  # triggers the Polytope request

hp.mollview(field.values, title="ICON — avg 2m temperature — Jun 2019",
            unit="K", cmap="RdYlBu_r", nest=True, flip='geo')
plt.show()

In [ ]:
#field2 = ds["avg_tprate"].sel(model="IFS-FESOM", time=slice("2014-01", "2014-12")).mean("time")
field2 = ds["avg_tprate"].sel(model="IFS-FESOM", time="2014-07-01")

hp.mollview(field2.values,
            title="IFS-FESOM — avg total precipitation — July 2014",
            unit="m", cmap="Blues", min=0, max=0.0001, nest=True, flip='geo')
plt.show()

## 4. Compare experiments: climate change signal

Create a second store for the scenario experiment, then take the difference.
Each `.sel()` triggers a single Polytope fetch.

In [ ]:
SCEN_EXPERIMENT = "SSP3-7.0"
SCEN_YEARS      = range(2015, 2050)

scen_coords = {
    "time":  pd.date_range(f"{min(SCEN_YEARS)}-01", f"{max(SCEN_YEARS)}-12", freq="MS"),
    "cell":  range(N_CELLS),
    "model": MODELS,
}
if lt["levels"] is not None:
    scen_coords["level"] = lt["levels"]

scen_store = PolytopeZarrStore(
    address=ADDRESS,
    collection="destination-earth",
    base_request={
        "class": "d1",
        "dataset": "climate-dt",
        "type": "fc",
        "expver": "0001",
        "generation": "2",
        "realization": "1",
        "activity": "projections",
        "experiment": SCEN_EXPERIMENT,
        "levtype": lt["levtype"],
        "resolution": RESOLUTION,
        "stream": "clmn",
    },
    coords=scen_coords,
    variables=lt["variables"],
    internal_dims=["cell"],
    time_fields=["year", "month"],
)

ds_scen = scen_store.open()
ds_scen

In [ ]:
# Single-month comparison: July 2014 (hist) vs July 2030 (scenario)
hist_field = ds["avg_2t"].sel(model="ICON", time="2014-07-01")
scen_field = ds_scen["avg_2t"].sel(model="ICON", time="2030-07-01")

diff = scen_field.values - hist_field.values

hp.mollview(diff,
            title="ICON — ΔT (SSP3-7.0 Jun 2030 minus hist Jun 2019)",
            unit="K", cmap="RdBu_r", min=-3, max=3, nest=True, flip='geo')
plt.show()

## Notes for further use

- **One Polytope request per field** (1 model × 1 month × all cells).
  This is great for browsing; for bulk 30-year means, better use `02_climate_change_destine.ipynb`
  which batches multiple years per request.
- Try the other `LEVTYPES` as well in order to browse the full Climate DT portfolio, i.e. `o2d/o3d`, `pl/hl`, or `sol`
- `store.clear_cache()` frees memory from previously fetched fields.

In [ ]:
# Free memory if needed
store.clear_cache()
scen_store.clear_cache()